# 4 – RAG with Late Chunking

**The problem** (look at `data/novaterra_annual_report.txt`):

> "The company's revenue rose to EUR 4.2 billion in 2025 ..."

That chunk never says "Novaterra". With normal ("early") chunking each chunk is embedded **alone**, so its vector knows it is about *some company's revenue* – but not which one. A question about Novaterra's revenue may then match Meridian Bank or Harbor City chunks just as well.

**Late chunking** (Günther et al., 2024, Jina AI, [arXiv:2409.04701](https://arxiv.org/abs/2409.04701)) swaps the order of two steps:

```
Early chunking:  split text -> embed each chunk separately -> pool
Late chunking:   embed WHOLE document -> split token vectors -> pool
```

```
whole document ──> long-context transformer ──> one vector per TOKEN
                                                 │   │   │   │   │   │
                                 chunk spans:    [ chunk 1 ][ chunk 2 ]
                                                 │           │
                                          mean-pool     mean-pool
                                                 ▼           ▼
                                              vector 1    vector 2
```

Because self-attention ran over the whole document, the token vectors of "The company's" already carry information from the title "Novaterra Energy AG". Mean-pooling them per chunk gives chunk vectors that are *context-aware*, while the chunk text we store and show the LLM stays exactly the same.

The next section explains what happens inside an embedding model and derives the three requirements late chunking puts on it – **mean pooling**, **access to the token-level output** and a **long context window** – and why they rule out Ollama's embedding API for this one example (the LLM still runs on Ollama).

LangChain is still used for loading, splitting, the vector store and the LLM. We write our own vectors straight into Chroma and then query it through LangChain's `Chroma` wrapper as usual.

## Background: how an embedding model turns text into one vector

Every transformer embedding model works in four stages. Once you see them, late chunking is almost obvious.

1. **Tokenize.** The text is cut into tokens (sub-word pieces) and each token becomes an integer id. `"The company's revenue rose to EUR 4.2 billion."` becomes 14 tokens, including a `[CLS]` token at the start and a `[SEP]` token at the end that the tokenizer adds itself. A *fast* tokenizer can also report which characters of the text each token covers – we will need that.
2. **Encode.** Every layer of the transformer updates every token's vector by *self-attention*: the new vector of a token is a weighted mix of the vectors of **all** tokens in the input. After the last layer there is one vector per token – 768 numbers each for ModernBERT-base – and each of those vectors already "knows" about the rest of the input. This matrix of shape `(num_tokens, 768)` is the model's `last_hidden_state`. We call it the **token-level output**.
3. **Pool.** A vector database wants one vector per text, not one per token. *Pooling* collapses the matrix into a single vector. The common choices are **mean pooling** (average all token vectors), **CLS pooling** (keep only the first token's vector) and **last-token pooling** (keep the last token's vector; typical for decoder-style embedders).
4. **Normalize.** The vector is scaled to length 1, so that the dot product of two vectors is their cosine similarity.

`sentence-transformers`, LangChain's embedding classes and Ollama's `/api/embed` run all four stages inside one call and hand back only the result of stage 4. The recipe is stored with the model: for `nomic-ai/modernbert-embed-base`, `modules.json` lists `Transformer → Pooling → Normalize`, and `1_Pooling/config.json` sets `pooling_mode_mean_tokens: true`.

### Where early chunking loses the context

With early chunking, stage 2 only ever sees the tokens of one chunk. If the chunk says "The company's revenue rose …" and the word "Novaterra" sits in another chunk, no token vector of this chunk can contain any trace of "Novaterra". The information is not lost during pooling – it was never computed. Chunk overlap, prepending the title to every chunk, or the propositions of Example 2 are all workarounds that push the context back *into the text*.

Late chunking fixes it at stage 2: run the transformer over the **whole document**, so that "company's" attends to "Novaterra Energy AG" in the title and the first paragraph, and only *then* cut. The cutting happens on the token-level output: chunk 1's vector is the mean of chunk 1's rows, chunk 2's vector is the mean of chunk 2's rows, and so on. The chunk *texts* are unchanged; only their vectors become context-aware.

### Why we need access to the token-level output

Late chunking is "late" because pooling happens *after* chunking. That is only possible if we can get hold of the stage-2 matrix before anybody pools it. A pooled vector cannot be un-pooled: the mean of 500 token vectors is one vector, and the 500 originals are gone. So an API that returns one vector per text – which is every embedding API – can only give us early chunking (send each chunk) or one vector for the whole document (send the document). Neither is what we want.

Therefore we load the model with `transformers`' `AutoModel`, whose output *is* the token matrix, and do stages 3 and 4 ourselves. We also need the character offsets from stage 1, to translate "this chunk is characters 704–1120 of the document" into "this chunk is rows 143–231 of the matrix".

### Why the model must use mean pooling

Mean pooling has a property the other pooling methods lack: the pooled vector of *any span* of tokens is simply the average of that span's rows. That makes "pool later, per chunk" a cheap and well-defined operation on a matrix we already have – one forward pass per document instead of one per chunk.

- With **CLS pooling** the model is trained to compress the whole input into the single `[CLS]` vector. There is one `[CLS]` per document, not one per chunk, so there is nothing to pool per chunk. Averaging the other token vectors of a CLS-trained model does not help: those vectors were never trained to be meaningful when averaged.
- With **last-token pooling** the summary sits in the final token, which again exists once per input.

There is a second, subtler reason. The query is embedded the normal way (whole-text mean pooling) and compared to the chunk vectors by cosine similarity. That comparison only makes sense if query and chunk vectors live in the same space, which they do when both are means of the *same model's* token vectors. For a model trained with mean pooling, the average of a span of token vectors is the same kind of object the model would produce for that span as a stand-alone text; our late-chunked vectors are therefore legitimate embeddings, just with more context inside them.

### Why we need a long context window

Stage 2 must see the whole document in one forward pass. ModernBERT accepts 8,192 tokens; classic BERT-style embedders stop at 512, a few paragraphs. For documents longer than the window, real systems cut them into large "macro chunks" (e.g. 8k tokens with some overlap) and late-chunk inside each. The cost of self-attention grows quadratically with the sequence length, so one 8k-token pass is more expensive than ten 800-token passes – that is the price of the context.

## Imports

- `os`, `pathlib.Path` – the `RAG_LLM` environment variable and file paths.
- `chromadb` – the *native* Chroma client. In Examples 1–3 LangChain's `Chroma` class called the embedding model for us; here we compute the vectors ourselves and must insert them explicitly, which the native client's `collection.add(embeddings=...)` allows.
- `torch`, `torch.nn.functional as F` – the model returns PyTorch tensors. We use boolean indexing and `.mean()` on them, `F.normalize` for the unit-length step, and `torch.no_grad()` to switch off gradient bookkeeping.
- `init_chat_model` – the chat model, served by Ollama as in the other examples.
- `Chroma`, `Document`, `ChatPromptTemplate`, `StrOutputParser`, `RecursiveCharacterTextSplitter` – the same LangChain pieces as before.
- `Embeddings` – LangChain's abstract base class for embedding models. It has two methods, `embed_documents` and `embed_query`; we implement it in a few lines so that LangChain's `Chroma` can embed *queries* with our model.
- `AutoTokenizer`, `AutoModel` – Hugging Face `transformers`. The `Auto*` classes read a model's `config.json` from the Hub and instantiate the matching architecture (`ModernBertModel`) and tokenizer. `AutoModel` is the *bare* encoder without any task head, so what it returns is the token-level hidden states we need.

In [ ]:
import os
from pathlib import Path

import chromadb
import torch
import torch.nn.functional as F
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoModel, AutoTokenizer

## Configuration

- `LLM_MODEL` – the Ollama-served chat model, as in the other examples.
- `EMBEDDING_MODEL` – `nomic-ai/modernbert-embed-base`. It satisfies all three requirements from the background section: an 8,192-token context window, mean pooling (see its `1_Pooling/config.json`), and – like any Hub model – it can be loaded with `AutoModel` to get the token-level output. 22 transformer layers, 768-dimensional vectors, ~150M parameters, ~600 MB download. Jina's `jinaai/jina-embeddings-v2-small-en` from the original paper also works but needs `trust_remote_code=True`.
- `MAX_TOKENS = 8192` – the model's `max_position_embeddings`. Used twice: to truncate inputs that would not fit, and to detect in `late_chunk_embeddings` that truncation happened. Our three documents are 350–510 tokens long, far below the limit.
- `DOC_PREFIX` / `QUERY_PREFIX` – the model was trained *asymmetrically*: documents prefixed with `search_document: `, queries with `search_query: `. The prefixes tell the model which role a text plays, so that the two kinds of vectors are comparable; without them retrieval quality drops. In late chunking the document prefix is added **once**, in front of the whole document, not per chunk. That is why the code later shifts every chunk's character offset by `len(DOC_PREFIX)` (17 characters), and why the prefix tokens belong to no chunk.
- `CHUNK_SIZE = 400` characters (roughly 80–100 tokens) – deliberately small, so that most chunks are a single paragraph that does not name its subject. That is the situation late chunking was designed for.
- `CHUNK_OVERLAP = 0` – overlap is a workaround for lost context, and late chunking is supposed to make it unnecessary. Zero overlap also keeps the naive/late comparison clean: both indices hold exactly the same texts.
- `TOP_K = 3` – how many chunks we show per index and hand to the LLM.

In [ ]:
DATA_DIR = Path("data")
LLM_MODEL = os.getenv("RAG_LLM", "ollama:mistral-small3.2:24b")

EMBEDDING_MODEL = "nomic-ai/modernbert-embed-base"
MAX_TOKENS = 8192

DOC_PREFIX = "search_document: "
QUERY_PREFIX = "search_query: "

CHUNK_SIZE = 400
CHUNK_OVERLAP = 0
TOP_K = 3

print("LLM:", LLM_MODEL)

## The embedding model, used through `transformers` directly

`AutoTokenizer.from_pretrained(EMBEDDING_MODEL)` downloads the tokenizer files (vocabulary and merge rules) from the Hub and returns a *fast*, Rust-backed tokenizer. "Fast" matters here: only fast tokenizers can return the character offsets of each token (`return_offsets_mapping=True`), which late chunking needs to map chunks onto rows of the token matrix.

`AutoModel.from_pretrained(EMBEDDING_MODEL)` downloads the weights (cached in `~/.cache/huggingface`) and builds the encoder: an embedding layer followed by 22 transformer layers. `AutoModel` deliberately has no task head, so calling it returns the last layer's hidden state for every token – stage 2 of the background section, nothing more. `.eval()` switches the model to inference mode (dropout off), so the same text always gives the same vectors. Everything runs on the CPU here, which is fine for three short documents.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
model = AutoModel.from_pretrained(EMBEDDING_MODEL).eval()

### What the model returns: one vector per token

Before building anything, look at the raw output for one sentence. We tokenize with `return_offsets_mapping=True`, run the model once, and print the shape of `last_hidden_state` together with the characters each token covers. Three things to notice:

1. The shape is `(1, 14, 768)` = (batch, tokens, hidden size): one 768-dimensional vector **per token**, nothing pooled yet. This is the matrix every embedding API throws away after pooling.
2. `[CLS]` and `[SEP]` were added by the tokenizer and have an empty `(0, 0)` span – they correspond to no characters.
3. Tokens carry their leading space: ` company` covers characters 3–11, including the space in front of it (the `Ġ` in the raw token name is how this tokenizer family marks that space). This detail matters when we decide which tokens belong to which chunk.

`offset_mapping` has to be removed from `enc` before the model is called – the model does not accept it as an argument.

In [ ]:
sample = "The company's revenue rose to EUR 4.2 billion."
enc = tokenizer(sample, return_tensors="pt", return_offsets_mapping=True)
offsets = enc.pop("offset_mapping")[0]
with torch.no_grad():
    hidden = model(**enc).last_hidden_state

print("last_hidden_state:", tuple(hidden.shape), "= (batch, tokens, hidden size)\n")
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
for i, (token, (start, end)) in enumerate(zip(tokens, offsets.tolist())):
    print(f"{i:2d}  {token:12}  characters {start:2d}-{end:2d}  {sample[start:end]!r}")

### Classic ("early") embedding

`mean_pool(token_vectors, mask)` is stages 3 and 4 of the background section, written so that it works for *any* selection of tokens:

- `token_vectors` has shape `(num_tokens, 768)`; `mask` is a boolean tensor with one entry per token.
- `token_vectors[mask]` keeps only the rows where the mask is `True` → `(num_selected, 768)`.
- `.mean(dim=0)` averages those rows → one vector of 768 numbers.
- `F.normalize(pooled, dim=0)` divides the vector by its length → unit length. Cosine similarity is then a plain dot product, and our vectors match what the model's standard pipeline produces (its last module is `Normalize`).

The function does not care *which* tokens the mask selects – that is the whole trick. For a normal embedding the mask is "all real tokens"; for late chunking it will be "the tokens of this chunk".

`embed_text(text)` is the complete standard pipeline, used for queries and for the naive baseline:

- `tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_TOKENS)` returns a dict with `input_ids` and `attention_mask` as tensors of shape `(1, num_tokens)`. `[CLS]` and `[SEP]` are added automatically; inputs longer than `MAX_TOKENS` are cut.
- `model(**enc).last_hidden_state` is the forward pass → `(1, num_tokens, 768)`; `[0]` picks the only text in the batch → `(num_tokens, 768)`.
- `enc["attention_mask"][0].bool()` is `True` for real tokens and `False` for padding. With a single text there is no padding, so this selects every token, including `[CLS]` and `[SEP]` – exactly what `sentence-transformers`' mean pooling does for this model.
- `.tolist()` converts the tensor into a plain Python list of floats, which is what Chroma and LangChain expect.
- `@torch.no_grad()` tells PyTorch not to build the computation graph that training would need; less memory and faster.

This function reproduces `SentenceTransformer(EMBEDDING_MODEL).encode(text)` up to floating-point noise. We write it out ourselves so that the late version below can reuse `mean_pool` with a different mask.

In [ ]:
def mean_pool(token_vectors, mask):
    pooled = token_vectors[mask].mean(dim=0)
    return F.normalize(pooled, dim=0)


@torch.no_grad()
def embed_text(text):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_TOKENS)
    token_vectors = model(**enc).last_hidden_state[0]
    mask = enc["attention_mask"][0].bool()
    return mean_pool(token_vectors, mask).tolist()

### A minimal LangChain `Embeddings` wrapper

LangChain's `Embeddings` interface has two methods: `embed_documents(texts) -> list[list[float]]` and `embed_query(text) -> list[float]`. Our class adds the right prefix and delegates to `embed_text`.

- `embed_query` is what LangChain's `Chroma` calls for every `similarity_search`. Queries are always embedded the normal way: a query is one short text with no surrounding document, so there is nothing to late-chunk. Its vector lives in the same space as our chunk vectors because it is the mean of the same model's token vectors.
- `embed_documents` is the naive baseline: every chunk gets the document prefix and is embedded **in isolation**. We call it ourselves when building the naive index; LangChain never calls it in this notebook, because we insert the vectors into Chroma by hand.

In [ ]:
class ModernBertEmbeddings(Embeddings):
    def embed_query(self, text):
        return embed_text(QUERY_PREFIX + text)

    def embed_documents(self, texts):
        return [embed_text(DOC_PREFIX + t) for t in texts]

## LATE CHUNKING – the core of this example

`late_chunk_embeddings(document_text, chunks)` runs the transformer once over the whole document and then pools the token vectors per chunk. `chunks` must carry `metadata["start_index"]`, the character offset of each chunk inside `document_text` (the splitter adds it with `add_start_index=True`). Line by line:

1. **Prefix once.** `full_text = DOC_PREFIX + document_text` puts the task prefix in front of the whole document, not in front of every chunk. All positions in `full_text` are therefore `shift = len(DOC_PREFIX)` (17) characters larger than the splitter's `start_index` values, which refer to `document_text`.
2. **Tokenize with offsets.** `return_offsets_mapping=True` adds `offset_mapping` to the result, shape `(1, num_tokens, 2)`: token *i* covers `full_text[offsets[i, 0] : offsets[i, 1]]`. `enc.pop("offset_mapping")` takes it out of the dict (the model would reject it as an argument) and `[0]` drops the batch dimension → `(num_tokens, 2)`. `truncation=True, max_length=MAX_TOKENS` cuts documents that do not fit the context window.
3. **One forward pass.** `token_vectors = model(**enc).last_hidden_state[0]` → `(num_tokens, 768)`. For the annual report that is 511 rows, and every row was computed by attention over all 511 tokens. This is the line where the context flows into every token; everything after it is bookkeeping.
4. **Truncation guard.** `enc["input_ids"].shape[1]` is the token count after truncation. If it reached `MAX_TOKENS`, the chunks beyond the cut-off have no tokens at all: their mask would be empty, the mean of nothing is `NaN`, and the vector is useless. Real systems avoid this by splitting very long documents into large "macro chunks" (e.g. 8k tokens with overlap) and late-chunking inside each.
5. **Select the chunk's tokens.** `start` and `end` are the chunk's character span in `full_text`; chunk texts are verbatim slices of the document, so `len(chunk.page_content)` gives the end. `token_starts` and `token_ends` are the two columns of `offsets`. The mask keeps every token whose span **overlaps** the chunk's span – `(token_ends > start) & (token_starts < end)` – and `(token_ends > token_starts)` drops `[CLS]` and `[SEP]`, whose span is empty.

   Why "overlaps" rather than "lies fully inside"? Because tokens carry their leading space (see the inspection above). The first token of a chunk that follows a space starts one character *before* the chunk; a "fully inside" test would silently drop it – in our corpus that is the first word of every document, right after the prefix, e.g. ` Novaterra`. With the separators we use, chunk boundaries fall on whitespace, so no token overlaps two chunks; the newline tokens between paragraphs and the prefix tokens overlap no chunk and are left out.
6. **Pool.** `mean_pool(token_vectors, in_chunk)` averages only the selected rows and normalises – the same function as in `embed_text`, just with a different mask – and `.tolist()` makes it a plain list.

Cost: one forward pass per document instead of one per chunk. Attention is quadratic in the sequence length, so a single long pass is more expensive than many short ones; that is the trade-off for the context.

In [ ]:
@torch.no_grad()
def late_chunk_embeddings(document_text, chunks):
    full_text = DOC_PREFIX + document_text
    shift = len(DOC_PREFIX)

    enc = tokenizer(
        full_text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=MAX_TOKENS,
    )
    offsets = enc.pop("offset_mapping")[0]
    token_vectors = model(**enc).last_hidden_state[0]

    if enc["input_ids"].shape[1] >= MAX_TOKENS:
        print("  WARNING: document truncated - later chunks get no vectors")

    vectors = []
    for chunk in chunks:
        start = chunk.metadata["start_index"] + shift
        end = start + len(chunk.page_content)
        token_starts, token_ends = offsets[:, 0], offsets[:, 1]
        in_chunk = (
            (token_ends > start)
            & (token_starts < end)
            & (token_ends > token_starts)
        )
        vectors.append(mean_pool(token_vectors, in_chunk).tolist())
    return vectors

## Indexing – two sets of vectors for the SAME chunks

- The splitter is the one from Examples 1–3 with one addition, `add_start_index=True`: every chunk gets `metadata["start_index"]`, its character position inside the document. `late_chunk_embeddings` needs it to find the chunk's rows in the token matrix. We split one `Document` at a time, so the indices refer to that document.
- `embeddings = ModernBertEmbeddings()` is our wrapper. It is used here for the naive vectors and later, inside Chroma, for the queries.
- For every file we compute two vectors per chunk:
  - **(a) early / naive chunking** – `embeddings.embed_documents([...])`: one forward pass *per chunk*, each chunk alone with its own prefix;
  - **(b) late chunking** – `late_chunk_embeddings(doc.page_content, chunks)`: one forward pass *per document*, then per-chunk pooling.
- `texts`, `metadatas`, `naive_vecs_all` and `late_vecs_all` are kept aligned position by position: chunk *i* is described by `texts[i]`, `naive_vecs_all[i]` and `late_vecs_all[i]`. The two indices will therefore contain exactly the same texts and metadata, and the only thing that differs is the vectors. That isolates the effect of late chunking.

Expect 5 + 7 + 8 = 20 chunks.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", r"(?<=\. )", " ", ""],
    is_separator_regex=True,
    add_start_index=True,
)
embeddings = ModernBertEmbeddings()

texts, metadatas, naive_vecs_all, late_vecs_all = [], [], [], []
for path in sorted(DATA_DIR.glob("*.txt")):
    doc = Document(page_content=path.read_text(encoding="utf-8"),
                   metadata={"source": path.name})
    chunks = splitter.split_documents([doc])

    naive_vecs = embeddings.embed_documents([c.page_content for c in chunks])
    late_vecs = late_chunk_embeddings(doc.page_content, chunks)

    texts += [c.page_content for c in chunks]
    metadatas += [c.metadata for c in chunks]
    naive_vecs_all += naive_vecs
    late_vecs_all += late_vecs
    print(f"  {path.name}: {len(chunks)} chunks")

### Writing the vectors into Chroma

Normally `Chroma.from_documents()` calls the embedding model for us. Here we already have the vectors, so we add them with the native `chromadb` client and then wrap each collection in LangChain's `Chroma` class.

- `chromadb.EphemeralClient()` is an in-process, in-memory Chroma; use `PersistentClient(path=...)` to keep the index on disk.
- `make_index(name, vectors)` first deletes a collection of the same name if it exists, so the cell can be re-run safely. This is needed because Chroma silently ignores `add` calls with ids it already knows, which would otherwise leave the old vectors in place after you change `CHUNK_SIZE` or the pooling code.
- `client.create_collection(name, metadata={"hnsw:space": "cosine"})` creates the collection and tells the HNSW index to use cosine distance (the default would be squared L2).
- `collection.add(ids=..., documents=texts, embeddings=vectors, metadatas=metadatas)` inserts everything in one call. Because `embeddings` is passed explicitly, Chroma stores the vectors as they are and never calls an embedding function. `ids` must be unique strings within the collection; `documents` keeps the chunk texts so that search results can return them; `metadatas` must be flat dicts of strings, numbers or booleans (ours hold `source` and `start_index`).
- `Chroma(client=client, collection_name=name, embedding_function=embeddings)` is LangChain's wrapper around the existing collection. The `embedding_function` is still needed for the **queries**: `similarity_search_with_relevance_scores(q)` calls `embeddings.embed_query(q)` – which adds `search_query: ` and runs `embed_text` – sends the vector to Chroma and converts the returned cosine distance into a similarity, `1 - distance`.

We build two collections, `naive` and `late`, over the same 20 texts.

In [ ]:
client = chromadb.EphemeralClient()


def make_index(name, vectors):
    if name in [c.name for c in client.list_collections()]:
        client.delete_collection(name)
    collection = client.create_collection(name, metadata={"hnsw:space": "cosine"})
    collection.add(
        ids=[f"{name}-{i}" for i in range(len(texts))],
        documents=texts,
        embeddings=vectors,
        metadatas=metadatas,
    )
    return Chroma(client=client, collection_name=name,
                  embedding_function=embeddings)


naive_index = make_index("naive", naive_vecs_all)
late_index = make_index("late", late_vecs_all)

## Retrieval comparison + generation

Both indices hold the same 20 texts in the same order, so any difference in ranking is caused by the vectors alone.

- `PROMPT` is the grounded prompt from the other examples: answer only from the context, say so if the answer is not there, cite the chunks used.
- `show(label, results)` prints each hit's rank, cosine similarity, source file (without extension) and the first 70 characters of the chunk, so that both rankings can be read side by side.

The questions are chosen so that the answer sits in a chunk that never names its entity: "The company's revenue rose …" and "The board proposes a dividend …" are in the same Novaterra chunk that only says "the company"; "a monthly pass costs 79 dollars" is in a Harbor City chunk that only says "the authority". What to expect: under naive chunking, the chunks that literally contain "Novaterra" (the title, "Novaterra Energy AG is a utility …") tend to outrank the answer chunk, because the entity name is the strongest signal in the query and the answer chunk's vector knows nothing about it. Under late chunking the answer chunk moves to the top and its cosine score rises, because its tokens absorbed "Novaterra" from the title and first paragraph during the whole-document forward pass.

In [ ]:
PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions using ONLY the context below. "
     "If the context does not contain the answer, say you don't know. "
     "Cite the chunks you used like [1], [2].\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])


def show(label, results):
    print(f"  {label}:")
    for rank, (doc, score) in enumerate(results, 1):
        source = Path(doc.metadata["source"]).stem
        preview = doc.page_content[:70].replace("\n", " ")
        print(f"    {rank}. cos={score:.3f}  [{source}]  {preview}...")

### Running the comparison

For every question:

1. `naive_index.similarity_search_with_relevance_scores(q, k=TOP_K)` and the same call on `late_index` each embed the question once through `embed_query`, run a cosine search in their collection and return `(Document, score)` pairs, best first.
2. The context for the LLM is built from the **late** results. Each chunk is numbered and labelled with its source file name: the chunk text itself still says only "the company", so the LLM needs to know which document it came from – late chunking fixes the vectors, not the text.
3. `PROMPT | llm | StrOutputParser()` fills the prompt, calls the Ollama model (`temperature=0` for deterministic answers) and returns the answer as a plain string.

In [ ]:
QUESTIONS = [
    "What was Novaterra's revenue in 2025?",
    "What dividend does Novaterra propose?",
    "How much does a monthly pass cost in Harbor City?",
]

llm = init_chat_model(LLM_MODEL, temperature=0)
for q in QUESTIONS:
    print(f"\nQuestion: {q!r}")
    show("Naive chunking", naive_index.similarity_search_with_relevance_scores(q, k=TOP_K))
    late_results = late_index.similarity_search_with_relevance_scores(q, k=TOP_K)
    show("Late chunking ", late_results)

    context = "\n\n".join(
        f"[{i}] (source: {Path(doc.metadata['source']).name})\n{doc.page_content}"
        for i, (doc, _) in enumerate(late_results, 1)
    )
    answer = (PROMPT | llm | StrOutputParser()).invoke(
        {"context": context, "question": q}
    )
    print("\nAnswer (late chunking context):", answer)
    print("-" * 80)

## Things to try

1. Ask about a Meridian Bank fact. Does late chunking help as much when every chunk already names its entity?
2. Set `CHUNK_SIZE` to 1000 and re-run the indexing cells. Does the gap between naive and late chunking shrink?
3. Compare the naive and late vectors of the same chunk directly: `torch.dot(torch.tensor(naive_vecs_all[i]), torch.tensor(late_vecs_all[i]))`. How different are they, and for which chunks is the difference largest?
4. Verify the pooling claim: `from sentence_transformers import SentenceTransformer` and compare `SentenceTransformer(EMBEDDING_MODEL).encode(DOC_PREFIX + text)` with `embed_text(DOC_PREFIX + text)`. The cosine similarity should be 1.0.
5. Give the naive side a `CHUNK_OVERLAP` of 100 characters. Does overlap close the gap to late chunking, or does the entity name still not reach the answer chunk?